In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_df = spark.table("bronze.nyc_taxi_raw")

In [0]:
bronze_df.printSchema()

### Changing column data type

In [0]:
silver_df = bronze_df \
    .withColumn(
        "tpep_pickup_datetime",
        to_timestamp("tpep_pickup_datetime")
    ) \
    .withColumn(
        "tpep_dropoff_datetime",
        to_timestamp("tpep_dropoff_datetime")
    )

### Filtering data where trip_distance and fare_amount is greater than 0

In [0]:
silver_df = silver_df.filter(
    col("trip_distance") > 0
).filter(
    col("fare_amount") > 0
)

### Adding new column trip_duration_minutes, pickup_date, pickup_hour

In [0]:
silver_df = silver_df.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp("tpep_dropoff_datetime")
        - unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

In [0]:
silver_df = silver_df.withColumn(
    "pickup_date",
    to_date("tpep_pickup_datetime")
)

In [0]:
silver_df = silver_df.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

In [0]:
display(silver_df)

In [0]:
silver_df.printSchema()

### Creating Silver schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
(silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.nyc_taxi_clean"))

### Count of Null values in each column

In [0]:
null_counts = bronze_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bronze_df.columns
])

display(null_counts)

In [0]:
bronze_df.filter(
    ~col("vendor_id").isin("1", "2")
).display()

In [0]:
bronze_df.filter(
    col("tpep_pickup_datetime").isNull()
).display()

In [0]:
bronze_df.groupBy(bronze_df.columns) \
    .count() \
    .filter(col("count") > 1) \
    .display()

In [0]:
silver_df = silver_df.withColumn(
    "is_valid_trip",
    when(
        (col("trip_distance") > 0) &
        (col("fare_amount") > 0),
        True
    ).otherwise(False)
)

In [0]:
(silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.nyc_taxi_clean"))

### Changing order of columns

In [0]:
silver_df = silver_df.select(
    "vendor_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",

    # derived columns
    "trip_duration_minutes",
    "pickup_date",
    "pickup_hour",
    "is_valid_trip",

    # metadata columns 
    "ingestion_timestamp",
    "load_date",
    "source_file"
)

In [0]:
(silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.nyc_taxi_clean"))

In [0]:
%sql
SELECT *
FROM silver.nyc_taxi_clean
LIMIT 10;